In [0]:
import pyspark.sql.functions as F

In [0]:
bootstrap_servers = "pkc-7prvp.centralindia.azure.confluent.cloud:9092"
api_key = "LNEJDSOEA2WWNS4D"
api_secret = "cfltRveMISDzuQWmJEq8RLHDe88wdg6uISXFuI1YlDbkvw6wlcJz1PcbPjrIRiMg"
topic_name = "credit_card_transactions"

jass_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
sample_batch_df = (spark.read.format('kafka')
                   .option("kafka.bootstrap.servers", bootstrap_servers)
                   .option("subscribe", topic_name)
                   .option("kafka.security.protocol", "SASL_SSL")
                   .option("kafka.sasl.mechanism", "PLAIN")
                   .option("kafka.sasl.jaas.config", jass_config)
                #    .option("startingOffsets", "latest")
                   .load())

display(sample_batch_df)

In [0]:
print(sample_batch_df.count())
sample_batch_df.printSchema()


In [0]:
parsed_df = sample_batch_df.select(
    F.col("key").cast("string"),
    F.col("value").cast("string"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp")
)
display(parsed_df)
print(parsed_df.select('key').distinct().count())

In [0]:
# %sql
# CREATE CATALOG IF NOT EXISTS finuard_project;
# USE CATALOG finuard_project;
# CREATE SCHEMA IF NOT EXISTS test;

In [0]:
parsed_df.write.mode("overwrite").saveAsTable("finuard_project.test.transactions_batch")

In [0]:
%sql
select * from finuard_project.test.transactions_batch